In [2]:
# Imports
import pandas as pd
import numpy as np
from tabulate import tabulate
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, BayesianRidge, PoissonRegressor
from difflib import SequenceMatcher

import warnings
warnings.filterwarnings('ignore')

## Utility Functions

In [3]:
def load_node_profile(node):
    filepath = f'../../tool/infrastructure-profiler/profiles/{node}.csv'

    with open(filepath) as f:
        data = f.readlines()[1].split(',')

    return {
        'ram': int(data[1].strip()),
        'cores': int(data[2].strip()),
        'io_read': float(data[3].strip()),
        'io_write': float(data[4].strip()),
        'sysbench': float(data[5].strip()),
        'z7b': float(data[6].strip()),
        'bench_time': float(data[7].strip())
    }

def combine_df(input_df, trace_df):
    rows = []

    for name, group in input_df.groupby('name'):
        no_files = str(len(group))
        filepaths = group['path']
        paths = ','.join([filepath.split('/')[-1].split('.')[-1] for filepath in filepaths])
        total_size = group['size'].sum() / 1073741824  # convert from B to GiB

        if len(trace_df[trace_df['name'] == name]) > 0:
            task_record = trace_df[trace_df['name'] == name].iloc[0]
            row = [task_record['task_id'], task_record['name'], task_record['%cpu'], 
                task_record['start'], task_record['complete'], task_record['complete'] - task_record['start'], 
                no_files, paths, total_size, task_record['name'].split(':')[-1].split(' ')[0], task_record['hostname']]
            rows.append(row)
        else:
            print('err:', name)

    combined = pd.DataFrame(rows, columns=['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_gib', 'short_name', 'hostname'])
    return combined

def load_profiling_runs(node, workflows, samples, node_profiles):
    node_profile = node_profiles[node]
    runs = {}

    for workflow, samples in zip(workflows, samples):
        runs[workflow] = {}

        for sample in samples:
            sample_dir = f'{node}/{workflow}/{sample}/'
            input_df = pd.read_csv(sample_dir + 'input.csv', sep=';')
            trace_df = pd.read_csv(sample_dir + 'trace.csv')
            runs[workflow][sample] = combine_df(input_df, trace_df)

    return (node_profile, runs)

def load_workflow_execution(cluster, workflow, iteration):
    execution_dir = f'../{cluster}/{workflow}/{iteration}/'
    input_df = pd.read_csv(execution_dir + 'input.csv', sep=';')
    trace_df = pd.read_csv(execution_dir + 'trace.csv')
    return combine_df(input_df, trace_df)

def load_ichnos_trace(cluster, workflow, iteration, gov):
    short_cluster = cluster.split('-')[0]
    trace_loc = f'../{cluster}/{workflow}/ichnos/{short_cluster}-{workflow}-{iteration}-1-{gov}-trace.csv'
    return pd.read_csv(trace_loc)

def process_ichnos_summary(raw):
    core_total_energy = float(raw[7].split(': ')[-1][:-3])
    mem_dynamic_energy = float(raw[9].split(': ')[-1][:-3])
    mem_static_energy = float(raw[15].split(': ')[-1][:-3])
    return {
        'core_total': core_total_energy, 
        'mem_dynamic': mem_dynamic_energy, 
        'mem_static': mem_static_energy, 
        'total': core_total_energy + mem_dynamic_energy + mem_static_energy
    } 

def load_ichnos_summary(cluster, workflow, iteration, gov):
    short_cluster = cluster.split('-')[0]
    with open(f'../{cluster}/{workflow}/ichnos/{short_cluster}-{workflow}-{iteration}-1-{gov}-summary.txt', 'r') as file:
        raw = file.readlines()
    raw = [line.strip() for line in raw]
    return process_ichnos_summary(raw)

def load_rapl_summary(cluster, workflow, iteration):
    rapl_vals = pd.read_csv(f'../{cluster}/{workflow}-runs.csv')
    return rapl_vals.iloc[[iteration - 1]]

def load_power_model(node, setting=''):
    if setting != '':
        filename = f'{setting}-model.txt'
    else: 
        filename = 'model.txt'

    with open(f'../../tool/infrastructure-profiler/raw/{node}/{filename}', 'r') as f:
        data = [line.strip() for line in f.readlines()[1:]]

    coeff = float(data[0].split(',')[0][2:].strip())
    inter = float(data[0].split(',')[1][:-2].strip())
    draw = float(data[1].split(':')[1].strip())

    if draw == 0:
        draw = 0.392

    return (coeff, inter, draw)

def get_task_data_from_experiment(task_name, experiment):
    task_data = experiment[experiment['short_name'] == task_name]
    key_data = task_data[['input_gib', 'runtime', '%cpu']]
    key_data.columns = ['data', 'time', 'usage']
    key_data.reset_index().drop('index', axis=1)
    return key_data

def get_error(predicted, actual):
    return abs(actual - predicted) / actual * 100

In [4]:
class MedianModel:
    def __init__(self, median):
        self.median = median
    
    def predict(self, x):
        x = np.asarray(x)
        value = x.shape[0]
        return np.full(value, self.median)

## Setup Data

In [5]:
# Constants
HU = 'hu'
HU_40 = 'hu-c40'
HU_42 = 'hu-c42'
HU_44 = 'hu-c44'
HU_45 = 'hu-c45'
GPG = 'gpg'
GPG_13 = 'gpgnode-13'
GPG_14 = 'gpgnode-14'
GPG_15 = 'gpgnode-15'
GPG_16 = 'gpgnode-16'
GPG_17 = 'gpgnode-17'
GPG_18 = 'gpgnode-18'
GPG_22 = 'gpgnode-22'
AWS = 'aws'
AWS_c5n = 'aws-metal-c5n'
AWS_z1d = 'aws-metal-z1d'
AWS_c5d = 'aws-metal-c5d'
GCP_n2 = 'gcp-n2'
wfs = {
    GPG: ['rnaseq', 'chipseq', 'nanoseq'], 
    HU: ['rnaseq', 'chipseq', 'rangeland', 'sarek', 'nanoseq']
}
VR = 'VR'
RF = 'RandomForest'
LR = 'LinearRegression'
BR = 'BayesianRidge'
PR = 'PoissonRegressor'
GBR = 'GradientBoostingRegressor'

In [6]:
# Node Profiles + Power Models (todo)
nodes = [AWS_c5n, AWS_z1d, GPG_13, GPG_14, GPG_15, GPG_16, GPG_17, GPG_18, GPG_22, HU_40, HU_42, AWS_c5d, HU_44, HU_45, GCP_n2]
node_profiles = {}

for node in nodes:
    profile = load_node_profile(node)
    node_profiles[node] = profile

nodes_by_host = {
    'hu-worker-c40': HU_40,
    'hu-worker-c42': HU_42,
    'hu-worker-c44': HU_44,
    'hu-worker-c45': HU_45
}

gov_by_workflow = {
    GPG: {
        "rnaseq": "ondemand_linear",
        "chipseq": "ondemand_linear",
        "nanoseq": "ondemand_linear",
    },
    HU: {
        "rnaseq": "schedutil_linear",
        "chipseq": "schedutil_linear",
        "rangeland": "schedutil_linear",
        "sarek": "schedutil_linear",
        "nanoseq": "performance_linear",
    }
}

In [7]:
# Workflow Executions 
wf_executions = {}

for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    wf_executions[cluster] = {}
    for wf in wfs[cluster]:
        wf_executions[cluster][wf] = {}
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration] = {}
            wf_executions[cluster][wf][iteration]['run'] = load_workflow_execution(cluster_path, wf, iteration)

err: NFCORE_SAREK:SAREK:BAM_VARIANT_CALLING_TUMOR_ONLY_ALL:BAM_VARIANT_CALLING_SINGLE_STRELKA:MERGE_STRELKA (SRR11452475)


In [8]:
# Workflow RAPL Total
for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration]['rapl'] = load_rapl_summary(cluster_path, wf, iteration)

In [9]:
# Workflow Ichnos Total
for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration]['ichnos'] = load_ichnos_summary(cluster_path, wf, iteration, gov_by_workflow[cluster][wf])

In [10]:
# Join Workflow Traces with Ichnos Traces 
def add_energy_with_task_id(run, ichnos):
    combined = pd.merge(run, ichnos[['id', 'core_kwh', 'mem_kwh']], on='id').drop_duplicates()
    return combined[['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_gib', 'short_name', 'hostname', 'core_kwh', 'mem_kwh']]

# Workflow Ichnos Total
for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            run = wf_executions[cluster][wf][iteration]['run']
            ichnos_trace = load_ichnos_trace(cluster_path, wf, iteration, gov_by_workflow[cluster][wf])
            df_with_energy = add_energy_with_task_id(run, ichnos_trace)
            wf_executions[cluster][wf][iteration]['run'] = df_with_energy

In [11]:
# Compare RAPL Total to Ichnos Total
gu_socket = {'rnaseq': '_2', 'chipseq': '_2', 'nanoseq': 'both'}
hu_socket = {'rnaseq': '', 'chipseq': '', 'nanoseq': '', 'rangeland': '', 'sarek': ''}

for cluster, cluster_path, socket in zip([GPG, HU], ['gu-cluster', 'hu-cluster'], [gu_socket, hu_socket]):
    print('===', cluster, '===')
    for wf in wfs[cluster]:
        all_ichnos = []
        all_rapl = []

        for iteration in range(1, 4):
            ichnos_total = wf_executions[cluster][wf][iteration]['ichnos']['total']
            ichnos_core = wf_executions[cluster][wf][iteration]['ichnos']['core_total']
            ichnos_mem = wf_executions[cluster][wf][iteration]['ichnos']['mem_dynamic'] + wf_executions[cluster][wf][iteration]['ichnos']['mem_static']

            rapl_socket = socket[wf]
            if rapl_socket == 'both':
                rapl_core = wf_executions[cluster][wf][iteration]['rapl'][f'active_pkg_1'].iloc[0] + wf_executions[cluster][wf][iteration]['rapl'][f'active_pkg_2'].iloc[0]

                if wf_executions[cluster][wf][iteration]['rapl'][f'idle_dram_1'].iloc[0] + wf_executions[cluster][wf][iteration]['rapl'][f'idle_dram_2'].iloc[0] > wf_executions[cluster][wf][iteration]['rapl'][f'dram_1'].iloc[0] + wf_executions[cluster][wf][iteration]['rapl'][f'dram_2'].iloc[0]:
                    rapl_mem = wf_executions[cluster][wf][iteration]['rapl'][f'dram_1'].iloc[0] + wf_executions[cluster][wf][iteration]['rapl'][f'dram_2'].iloc[0]
                else:
                    rapl_mem = wf_executions[cluster][wf][iteration]['rapl'][f'active_dram_1'].iloc[0] + wf_executions[cluster][wf][iteration]['rapl'][f'active_dram_2'].iloc[0]
            else:
                rapl_core = wf_executions[cluster][wf][iteration]['rapl'][f'active_pkg{rapl_socket}'].iloc[0]

                if wf_executions[cluster][wf][iteration]['rapl'][f'idle_dram{rapl_socket}'].iloc[0] > wf_executions[cluster][wf][iteration]['rapl'][f'dram{rapl_socket}'].iloc[0]:
                    rapl_mem = wf_executions[cluster][wf][iteration]['rapl'][f'dram{rapl_socket}'].iloc[0]
                else:
                    rapl_mem = wf_executions[cluster][wf][iteration]['rapl'][f'active_dram{rapl_socket}'].iloc[0]

            rapl_total = rapl_core + rapl_mem
            wf_executions[cluster][wf][iteration]['rapl']['core'] = rapl_core
            wf_executions[cluster][wf][iteration]['rapl']['mem'] = rapl_mem
            core_error = abs(rapl_core - ichnos_core) / rapl_core * 100
            mem_error = abs(rapl_mem - ichnos_mem) / rapl_mem * 100
            total_error = abs(rapl_total - ichnos_total) / rapl_total * 100

            #print(f'{wf}-{iteration} ichnos: {ichnos_total:.2f}kWh rapl: {rapl_total:.2f}kWh error {total_error:.2f}%')
            all_ichnos.append(ichnos_total)
            all_rapl.append(rapl_total)
        avg_ichnos = np.mean(all_ichnos)
        avg_rapl = np.mean(all_rapl)

        print(f'{wf} i: {avg_ichnos:.2f}kWh, r: {avg_rapl:.2f}kWh, {get_error(avg_ichnos, avg_rapl):.2f}%')

=== gpg ===
rnaseq i: 1.52kWh, r: 1.43kWh, 6.99%
chipseq i: 2.00kWh, r: 1.90kWh, 5.64%
nanoseq i: 0.88kWh, r: 0.98kWh, 10.56%
=== hu ===
rnaseq i: 2.28kWh, r: 2.54kWh, 10.20%
chipseq i: 4.68kWh, r: 5.16kWh, 9.22%
rangeland i: 2.16kWh, r: 2.46kWh, 12.10%
sarek i: 4.83kWh, r: 5.28kWh, 8.53%
nanoseq i: 0.42kWh, r: 0.51kWh, 16.31%


In [51]:
# Load workflow profiling runs
wf_profiles = {}

# HU_40
wfs = ['rnaseq', 'chipseq', 'nanoseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], 
           ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j'],
           ['sample_k', 'sample_l', 'sample_m', 'sample_n', 'sample_o']]
profile, runs = load_profiling_runs(HU_40, wfs, samples, node_profiles)
wf_profiles[HU_40] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_40, 'schedutil')
}

# HU_42
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_g', 'sample_h', 'sample_i', 'sample_j']]  # sample_f to be added back
profile, runs = load_profiling_runs(HU_42, wfs, samples, node_profiles)
wf_profiles[HU_42] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_42, 'schedutil')
}

HU_44
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j']]  
profile, runs = load_profiling_runs(HU_44, wfs, samples, node_profiles)
wf_profiles[HU_44] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_44, 'schedutil')
}

HU_45
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j']]  
profile, runs = load_profiling_runs(HU_45, wfs, samples, node_profiles)
wf_profiles[HU_45] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_45, 'schedutil')
}

# gpgnode-13
wf_profiles[GPG_13] = {
    'node_profile': node_profiles[GPG_13],
    'power_model': load_power_model(GPG_13, 'ondemand')
}

# gpgnode-14
wf_profiles[GPG_14] = {
    'node_profile': node_profiles[GPG_14],
    'power_model': load_power_model(GPG_14, 'ondemand')
}

# gpgnode-15
wf_profiles[GPG_15] = {
    'node_profile': node_profiles[GPG_15],
    'power_model': load_power_model(GPG_15, 'ondemand')
}

# gpgnode-16
wf_profiles[GPG_16] = {
    'node_profile': node_profiles[GPG_16],
    'power_model': load_power_model(GPG_16, 'ondemand')
}

# gpgnode-17
wf_profiles[GPG_17] = {
    'node_profile': node_profiles[GPG_17],
    'power_model': load_power_model(GPG_17, 'ondemand')
}

# gpgnode-18
wf_profiles[GPG_18] = {
    'node_profile': node_profiles[GPG_18],
    'power_model': load_power_model(GPG_18, 'ondemand')
}

# AWS metal c5d
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_g', 'sample_h', 'sample_i', 'sample_j']]  # sample_f to be added back
profile, runs = load_profiling_runs(AWS_c5d, wfs, samples, node_profiles)
wf_profiles[AWS_c5d] = {
    'node_profile': node_profiles[AWS_c5d],
    'executions': runs,
    'power_model': load_power_model(AWS_c5d)
}

err: NFCORE_CHIPSEQ:CHIPSEQ:IGV
err: NFCORE_CHIPSEQ:CHIPSEQ:IGV


## Evaluation

In [27]:
# Augur Functions
def custom_fallback(task_name):
    #print(f"No match found for {task_name}, using default config") # todo @Kath !!!
    return ''

def find_closest_key(task_name, dictionary):
    task_parts = task_name.split('_')
    best_match = None
    best_score = 0

    for key in list(dictionary.keys()):
        key_parts = key.split('_')
        current_score = 0

        for pos, task_part in enumerate(task_parts):
            if task_part in key_parts:
                key_position = key_parts.index(task_part)
                position_weight = 1.0 / (pos + 1) + 1.0 / (key_position + 1)
                current_score += position_weight

        if task_parts[0] == key_parts[0]:
            current_score += 2.0  # first word importance        

        if current_score > best_score:
            best_score = current_score
            best_match = key 

    if best_match: 
        return best_match
    else: 
        return custom_fallback(task_name)

def compute_active_time_per_host(execution):
    tasks_by_host = {}

    for _, record in execution.iterrows():
        host = record['hostname']
        start_ts = record['start']
        end_ts = record['complete']

        if host in tasks_by_host:
            tasks_by_host[host].append((start_ts, end_ts))
        else:
            tasks_by_host[host] = [(start_ts, end_ts)]

    active_time_by_host = {}

    for host, intervals in tasks_by_host.items():
        if not intervals:
            continue

        intervals.sort(key=lambda x: x[0])  # sort by start timestamp
        merged = []
        current_start, current_end = intervals[0]

        for start, end in intervals[1:]:
            if start <= current_end:  # overlap or contiguous
                current_end = max(current_end, end)
            else:
                merged.append((current_start, current_end))
                current_start, current_end = start, end
        merged.append((current_start, current_end))

        total_ms: float = sum(end - start for start, end in merged)
        active_time_by_host[host] = total_ms / 1000 / 3600  # return time in hours

    return active_time_by_host

def reflector(cluster, task_name):
    # look at previous workflow traces -- for the same cluster, look for the matching name, train on data
    # if task name is not found, then fall back
    # train on workflow executions from the same cluster, with the same task name
    return []

# improve translation for runtime and for CPU usage 
def get_runtime_translation_factor(raw_factor, local_profile, target_profile):
    return (0.5 * raw_factor) + (0.5 * (local_profile['cores'] / target_profile['cores']))

def get_translation_factor(local_profile, target_profile):
    local_write_val = local_profile['io_write'] / local_profile['ram']
    local_read_val = local_profile['io_read'] / local_profile['ram']
    target_write_val = target_profile['io_write'] / target_profile['ram']
    target_read_val = target_profile['io_read'] / target_profile['ram']
    return (0.25 * local_profile['z7b'] / target_profile['z7b']) + \
    (0.25 * local_profile['sysbench'] / target_profile['sysbench']) + \
    (0.25 * local_write_val / target_write_val) + \
    (0.25 * local_read_val / target_read_val)
    #(0.25 * local_profile['bench_time'] / target_profile['bench_time']) + \

# ^ io + cpu 0.5 @Vasilis this is the function for translation factor, Lotaru does -- 
# ft = 0.5 * (cpu local / cpu_target) + 0.5 * (io local / io target)

In [14]:
# Prediction Model - Training
# @Vasilis - where we take profiling runs -> to make task predictions models and persist cpu usage
def train_reflector(runs, exclude_workflow):
    models = {}

    for workflow in runs.keys():
        if workflow == exclude_workflow:
            continue

        iterations = range(1, 4)
        tasks = runs[workflow][iterations[0]]['run']['short_name'].unique()

        for task in tasks:
            task_data = pd.DataFrame(columns=['data', 'time', 'usage'], dtype='float64')
            for iteration in iterations:
                experiment_data = get_task_data_from_experiment(task, runs[workflow][iteration]['run'])
                task_data = pd.concat([task_data, experiment_data])
            
            pearson = np.corrcoef(task_data['data'].to_numpy(), task_data['time'].to_numpy())[0, 1]
            models[task] = {}
            models[task]['usage'] = np.mean(task_data['usage'])

            if np.isnan(pearson) or pearson < 0.75:  
               models[task]['model'] = MedianModel(np.median(task_data['time'].to_numpy())) 
               models[task]['model_type'] = 'MedianModel'
            else:
                model = LinearRegression()
                models[task]['model_type'] = 'LinearRegression'
                models[task]['model'] = model.fit(task_data['data'].values.reshape(-1, 1), task_data['time'].values) 

    return models

def create_prediction_model(runs, model_type):
    models = {}

    for workflow in runs.keys():
        models[workflow] = {}
        samples = list(runs[workflow].keys()) 
        tasks = runs[workflow][samples[0]]['short_name'].unique() 

        for task in tasks:
            task_data = pd.DataFrame(columns=['data', 'time', 'usage'], dtype='float64')
            for sample in samples: 
                experiment_data = get_task_data_from_experiment(task, runs[workflow][sample])
                task_data = pd.concat([task_data, experiment_data])

            pearson = np.corrcoef(task_data['data'].to_numpy(), task_data['time'].to_numpy())[0, 1]
            models[workflow][task] = {}
            models[workflow][task]['usage'] = np.mean(task_data['usage'])

            if np.isnan(pearson) or pearson < 0.75:  
               models[workflow][task]['model'] = MedianModel(np.median(task_data['time'].to_numpy())) 
               models[workflow][task]['model_type'] = 'MedianModel'
               # this can be better, some tasks have a strong negative correlation?
               # deal with this special case !!
               # some tasks can use a different model?
               # @Vasilis - can check % of how many tasks scale linearly / not 
            else:
                if model_type == BR:
                    model = BayesianRidge(fit_intercept=True)
                    models[workflow][task]['model_type'] = 'BayesianRidge'
                elif model_type == GBR:
                    model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)
                    models[workflow][task]['model_type'] = 'GBR'
                else:
                    model = LinearRegression()
                    models[workflow][task]['model_type'] = 'LinearRegression'

                models[workflow][task]['model'] = model.fit(task_data['data'].values.reshape(-1, 1), task_data['time'].values) 

    return models

In [ ]:
# Prediction and Verification
headers = ['task', 'in (GiB)', 'pred_t (s)', 'actual_t (s)', 'error (%)', 'pred_u (%)', 'actual_u (%)', 'pred_cpu (kWh)', 'actual_cpu (kWh)', 'pred_mem (kWh)', 'actual_mem (kWh)', 'node']

def predict_wf(wf_model, workflow, local_profile, local_node = '', reflection_models = None): 
    data = []
    overall_error = []
    energy_error = []
    missing_tasks = [] 
    actual_execution = workflow['run']
    total_cpu_dynamic_energy = 0.0
    total_mem_dynamic_energy = 0.0
    reflector = reflection_models is not None

    for _, record in actual_execution.iterrows():
        task_name = record['short_name']
        data_size = record['input_gib']
        node = record['hostname']
        actual_runtime = record['runtime']
        actual_usage = record["%cpu"]

        if actual_runtime > 5000:  # exclude tasks that run for 0-5 seconds
            prediction_model = None

            if task_name not in wf_model.keys():
                if reflector:
                    if task_name in reflection_models.keys():
                        prediction_model = reflection_models[task_name]['model']
                        pred_usage = reflection_models[task_name]['usage']
                        task_name += '**REF'
                else:
                    missing_tasks.append((task_name))
                    continue
            else:
                model_name = task_name

            if prediction_model is None:
                prediction_model = wf_model[model_name]['model']
                pred_usage = wf_model[model_name]['usage']

            if node not in wf_profiles.keys():
                node = nodes_by_host[node]

            target_profile = wf_profiles[node]['node_profile']
            factor = get_translation_factor(local_profile, target_profile)
            factor = get_runtime_translation_factor(factor, local_profile, target_profile)
            target_power_model = wf_profiles[node]['power_model']

            if local_profile['cores'] != target_profile['cores']:
                predicted_runtime = prediction_model.predict([[data_size]])[0] * factor 
                cpu_usage = pred_usage * (local_profile['cores'] / target_profile['cores'])
            else:
                predicted_runtime = prediction_model.predict([[data_size]])[0] * factor  
                cpu_usage = pred_usage / target_profile['cores']

            if predicted_runtime > 0:
                error = (abs(predicted_runtime - actual_runtime) / actual_runtime) * 100 
                overall_error.append(error)

            # estimate energy consumption (ichnos logic)
            time_h = predicted_runtime / 1000 / 3600  # convert from ms to hours
            memory = data_size
            core_consumption = time_h * (target_power_model[0] * cpu_usage) * 0.001
            mem_consumption = memory * target_power_model[2] * time_h * 0.001
            total_cpu_dynamic_energy += core_consumption
            total_mem_dynamic_energy += mem_consumption
            predicted_energy = core_consumption + mem_consumption
            actual_energy = record['core_kwh'] + record['mem_kwh']

            if actual_energy > 0 and predicted_energy > 0:
                energy_error.append(get_error(predicted_energy, actual_energy))

            # report
            data.append([task_name, f'{data_size:.2f}', f'{predicted_runtime / 1000:.2f}', f'{actual_runtime / 1000:.2f}', error, f'{pred_usage:.2f}', f'{actual_usage}', f'{core_consumption}', f'{record['core_kwh']}', f'{mem_consumption}', f'{record['mem_kwh']}', node])

    task_runtime_error = np.mean(overall_error)
    task_energy_error = np.mean(energy_error)

    total_cpu_static_energy = 0.0
    total_mem_static_energy = 0.0
    active_time_per_host = compute_active_time_per_host(actual_execution)
    for host in active_time_per_host.keys():
        total_cpu_static_energy += active_time_per_host[host] * target_power_model[1] * 0.001  # convert to kWh
        total_mem_static_energy += active_time_per_host[host] * target_power_model[2] * target_profile['ram'] * 0.001 # convert to kWh

    augur_total_cpu = total_cpu_dynamic_energy + total_cpu_static_energy
    augur_total_mem = total_mem_dynamic_energy + total_mem_static_energy
    augur_total = augur_total_cpu + augur_total_mem
    ichnos_total = workflow['ichnos']['total']
    rapl_total = float(workflow['rapl']['core']) + float(workflow['rapl']['mem'])

    return ((augur_total_cpu, augur_total_mem, augur_total), ichnos_total, rapl_total, (task_runtime_error, task_energy_error, data, missing_tasks))

In [16]:
# @Vasilis
print(wf_profiles[HU_40]['executions']['rnaseq']['sample_a'])

    id                                               name   %cpu  \
0    2  NFCORE_RNASEQ:PREPARE_GENOME:CUSTOM_GETCHROMSI...   99.9   
1    1  NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom...   99.8   
2    6  NFCORE_RNASEQ:PREPARE_GENOME:MAKE_TRANSCRIPTS_...  100.1   
3    7  NFCORE_RNASEQ:PREPARE_GENOME:SALMON_INDEX (gen...  425.5   
4   16  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  135.8   
5   14  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  123.1   
6   15  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  120.3   
7   12  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  142.2   
8   10  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  563.6   
9    9  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:STAR_ALIGN_IGE...  274.3   
10  34  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...  132.2   
11  32  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...   95.0   
12  33  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...  119.6   
13  13  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_

In [62]:
# HU to GPG with and without reflector
hu_wf_models = create_prediction_model(wf_profiles[HU_40]['executions'], LR)
reflection_models = train_reflector(wf_executions[HU], 'rnaseq')
local_profile = wf_profiles[HU_40]['node_profile']
target_profile = wf_profiles[GPG_16]['node_profile']
target_power_model = wf_profiles[GPG_16]['power_model']
hu_rnaseq_model = hu_wf_models['rnaseq']
(_, _, _, ((task_runtime_error, task_energy_error, data, missing_tasks))) = predict_wf(hu_rnaseq_model, wf_executions[GPG]['rnaseq'][1], local_profile)

# @Vasilis (wip)
# why is the error inflated here -- well, some tasks randomly get predicted to be like 40000 seconds, when realtime is 40 s ?? 
# why is it so much larger, why these tasks? the actual runtime matches when we look at ms not s?? but i can't see why
# isolate the tasks below 
# names: CUSTOM_TX2GENE, TXIMETA_TXIMPORT
print('runtime', task_runtime_error, 'energy', task_energy_error, 'missing', missing_tasks)
print(tabulate(data, headers, tablefmt='orgtbl'))

runtime 255.7423344176932 energy 186.03237973663065 missing ['SAMTOOLS_SORT_QUALIMAP', 'SAMTOOLS_SORT_QUALIMAP', 'SAMTOOLS_SORT_QUALIMAP', 'SAMTOOLS_SORT_QUALIMAP', 'SAMTOOLS_SORT_QUALIMAP', 'SAMTOOLS_SORT_QUALIMAP']
| task                     |   in (GiB) |   pred_t (s) |   actual_t (s) |     error (%) |   pred_u (%) |   actual_u (%) |   pred_cpu (kWh) |   actual_cpu (kWh) |   pred_mem (kWh) |   actual_mem (kWh) | node       |
|--------------------------+------------+--------------+----------------+---------------+--------------+----------------+------------------+--------------------+------------------+--------------------+------------|
| CUSTOM_GETCHROMSIZES     |       2.93 |        15.72 |             19 |    17.2787    |       100.06 |           60.9 |      8.87117e-06 |        6.5271e-06  |      4.23964e-06 |        1.04912e-05 | gpgnode-18 |
| GTF_FILTER               |       3.78 |        28.54 |             30 |     4.8565    |        99.92 |           79.4 |      1.6088e-05 

In [61]:
# AWS to GPG
aws_wf_models = create_prediction_model(wf_profiles[AWS_c5d]['executions'], LR)
reflection_models = train_reflector(wf_executions[GPG], 'rnaseq')
local_profile = wf_profiles[AWS_c5d]['node_profile']
target_profile = wf_profiles[GPG_16]['node_profile']
target_power_model = wf_profiles[GPG_16]['power_model']
aws_rnaseq_model = aws_wf_models['rnaseq']
(_, _, _, ((task_runtime_error, task_energy_error, data, missing_tasks))) = predict_wf(aws_rnaseq_model, wf_executions[GPG]['rnaseq'][1], local_profile)

print('runtime', task_runtime_error, 'energy', task_energy_error)
print(tabulate(data, headers, tablefmt='orgtbl'))

# extra elevated task: SE_TRANSCRIPT_UNIFIED
# @Vasilis -- working on trying to predict runtime better, model seems to work better with the adjustment, but some tasks are wildly off, leading to the high error!
# i'm also ocnfused as to why our predicted energy is so off here when the runtime and usage is closer, looks like we predict more energy?

runtime 63.95171187791777 energy 3975.191462272848
| task                     |   in (GiB) |   pred_t (s) |   actual_t (s) |   error (%) |   pred_u (%) |   actual_u (%) |   pred_cpu (kWh) |   actual_cpu (kWh) |   pred_mem (kWh) |   actual_mem (kWh) | node       |
|--------------------------+------------+--------------+----------------+-------------+--------------+----------------+------------------+--------------------+------------------+--------------------+------------|
| CUSTOM_GETCHROMSIZES     |       2.93 |        14.21 |             19 |  25.2269    |       100.12 |           60.9 |      0.000770266 |        6.5271e-06  |      3.83228e-06 |        1.04912e-05 | gpgnode-18 |
| GTF_FILTER               |       3.78 |        36.54 |             30 |  21.7898    |        99.94 |           79.4 |      0.00197739  |        1.34367e-05 |      1.27245e-05 |        1.65651e-05 | gpgnode-18 |
| MAKE_TRANSCRIPTS_FASTA   |       3.71 |        93.36 |             96 |   2.74633   |       100

In [55]:
def predict_multiple_executions(local_node, local_profile, workflow, cluster, method):
    local_node_wf_models = create_prediction_model(wf_profiles[local_node]['executions'], method)
    local_wf_model = local_node_wf_models[workflow]
    # augur_results = []
    # ichnos_results = []
    # rapl_results = []
    # task_rt_results = []
    # task_en_results = []

    # for iteration in range(1, 4):
    #     (augur, ichnos, rapl, task_data) = predict_wf(local_wf_model, wf_executions[cluster][workflow][iteration], local_profile, None)
    #     augur_results.append(augur[2])
    #     ichnos_results.append(ichnos)
    #     rapl_results.append(rapl)
    #     task_rt_results.append(task_data[0])
    #     task_en_results.append(task_data[1])

    # augur_avg = np.mean(augur_results)
    # ichnos_avg = np.mean(ichnos_results)
    # rapl_avg = np.mean(rapl_results)
    # avg_rt_error = np.mean(task_rt_results)
    # avg_en_error = np.mean(task_en_results)

    #print(f'[original] {workflow},{local_node},{cluster},{avg_rt_error:.2f}%,{avg_en_error:.2f}%,{get_error(augur_avg, ichnos_avg):.2f}%,{get_error(augur_avg, rapl_avg):.2f}%')

    # AUGUR with REFLECTOR
    reflection_models = train_reflector(wf_executions[cluster], workflow)
    augur_results = []
    ichnos_results = []
    rapl_results = []
    task_rt_results = []
    task_en_results = []

    for iteration in range(1, 4):
        (augur, ichnos, rapl, task_data) = predict_wf(local_wf_model, wf_executions[cluster][workflow][iteration], local_profile, reflection_models)
        augur_results.append(augur[2])
        ichnos_results.append(ichnos)
        rapl_results.append(rapl)
        task_rt_results.append(task_data[0])
        task_en_results.append(task_data[1])

    augur_avg = np.mean(augur_results)
    ichnos_avg = np.mean(ichnos_results)
    rapl_avg = np.mean(rapl_results)
    avg_rt_error = np.mean(task_rt_results)
    avg_en_error = np.mean(task_en_results)

    print(f'[+reflect] {workflow},{local_node},{cluster},{avg_rt_error:.2f}%,{avg_en_error:.2f}%,{get_error(augur_avg, ichnos_avg):.2f}%,{get_error(augur_avg, rapl_avg):.2f}%')

In [56]:
# augur predictions from HU 40 - jupiter
local_node = HU_40
predict_multiple_executions(local_node, wf_profiles[local_node]['node_profile'], 'rnaseq', HU, LR)
predict_multiple_executions(local_node, wf_profiles[local_node]['node_profile'], 'chipseq', HU, LR)
predict_multiple_executions(local_node, wf_profiles[local_node]['node_profile'], 'nanoseq', HU, LR)
predict_multiple_executions(local_node, wf_profiles[local_node]['node_profile'], 'rnaseq', GPG, LR)
predict_multiple_executions(local_node, wf_profiles[local_node]['node_profile'], 'chipseq', GPG, LR)
predict_multiple_executions(local_node, wf_profiles[local_node]['node_profile'], 'nanoseq', GPG, LR)

[+reflect] rnaseq,hu-c40,hu,40.63%,56.26%,13.46%,22.29%
[+reflect] chipseq,hu-c40,hu,35.16%,53.34%,10.34%,18.60%
[+reflect] nanoseq,hu-c40,hu,42.62%,50.20%,30.13%,41.53%
[+reflect] rnaseq,hu-c40,gpg,252.22%,152.40%,9.72%,3.41%
[+reflect] chipseq,hu-c40,gpg,40.58%,77.44%,13.08%,8.17%
[+reflect] nanoseq,hu-c40,gpg,36.59%,85.26%,17.88%,26.55%


In [57]:
# predictions from AWS c5d - diana 
local_node = AWS_c5d
predict_multiple_executions(local_node, wf_profiles[local_node]['node_profile'], 'rnaseq', GPG, LR)
predict_multiple_executions(local_node, wf_profiles[local_node]['node_profile'], 'chipseq', GPG, LR)
predict_multiple_executions(local_node, wf_profiles[local_node]['node_profile'], 'rnaseq', HU, LR)
predict_multiple_executions(local_node, wf_profiles[local_node]['node_profile'], 'chipseq', HU, LR)

[+reflect] rnaseq,aws-metal-c5d,gpg,63.22%,2557.91%,395.27%,429.87%
[+reflect] chipseq,aws-metal-c5d,gpg,33.12%,3837.83%,709.93%,755.61%
[+reflect] rnaseq,aws-metal-c5d,hu,69.74%,5695.35%,1843.11%,1644.95%
[+reflect] chipseq,aws-metal-c5d,hu,62.02%,12446.73%,3756.79%,3401.38%
